# RotQuant Qwen3.5-4B — native GPU optimisation and GGUF comparison

## Goal
Keep the **W5/scale8 + W6-vocabulary model unchanged** while testing the
runtime. Select **A100 40GB**, then **Runtime → Run all**.

The previous pilot measured about **36.4 prefill / 19.8 decode tok/s**;
its 2,048-token stage stopped at an undersized three-minute cap. This
study uses context/repetition-aware budgets and selectable contexts.

Compare the original native kernel with an **experimental four-token
weight-reuse kernel**, then profile custom CUDA operators separately.
Pinned BF16 and Unsloth UD-Q4 GGUFs use the **same native bridge**, prompt
IDs, context reservation, FP16 cache, decode replay and timing rules.
No `llama-cpp-python` build and no new quantization/training are involved.

**No speedup is known yet.** The original kernel stays the default.
Correctness gates and stopped/partial-run labels are not weakened.

## Setup — explicit controls
Keep W6 alone for the first pilot. W8 is supported as a separate follow-up:
set `ARMS = ("b5_v8_s0",)` and use a new run name when ready.

`SOURCE_ROOT` must contain the original checkpoint, `prepared.json`,
`preparation.json`, and `packed_probes.safetensors` for each arm.
A reports-only ZIP is insufficient. The source is never modified.

The default **90 active-minute** allowance includes dependency setup,
hashing/copying, build and tests. Performance caps scale with context and
repetitions (default **4 / 6 / 12 minutes**). The 2 tok/s and 16 GiB limits are spending
guards, not pass marks for quality or production readiness.

In [ ]:
from pathlib import Path
import json, re, shutil, subprocess, sys

REPO_REF = "main"  # resolved once; use the published full commit for repeat runs
SOURCE_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_packed_validation/8f10ee60fc7f")
CACHE_ROOT = Path("/content/drive/MyDrive/rotquant/native_artifact_cache/v1")
ARMS = ("b5_v6_s0",)
CONTEXTS = (128, 512, 2048)  # use (2048,) for a targeted rerun with a new RUN_NAME
RUN_NAME = "study1"
ACTIVE_BUDGET_MINUTES = 90
BUILD_JOBS = 2
DECODE_STEPS = 32
MEASURED_REPETITIONS = 3  # plus one excluded warmup per context
MIN_DECODE_TOKENS_PER_SECOND = 2.0
MAX_PROCESS_VRAM_MIB = 16384

assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,63}", RUN_NAME)
assert 1 <= ACTIVE_BUDGET_MINUTES <= 180 and 1 <= BUILD_JOBS <= 32
assert ARMS and len(set(ARMS)) == len(ARMS) and set(ARMS) <= {"b5_v6_s0", "b5_v8_s0"}
BASELINES = ("bf16", "ud_q4")  # () explicitly skips these downloads/runs
RUN_PROFILE = True
assert len(set(BASELINES)) == len(BASELINES) and set(BASELINES) <= {"bf16", "ud_q4"}


### Connect Drive and pin the repository
This uses Colab's normal Drive mount; local `gws` authentication is not
required. The managed venv preserves Colab CUDA Torch and does not need
`ensurepip`, an external Hadamard kernel or `llama-cpp-python`.

Keep `CACHE_ROOT` private. Hashes detect accidental corruption, **not**
a malicious replacement of both cached binaries and their manifest.
Do not use downloaded/shared caches from an untrusted party.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert shutil.which("nvidia-smi") and shutil.which("nvcc"), "Select a CUDA GPU runtime."
subprocess.run(["nvidia-smi"], check=True, timeout=30)
for arm in ARMS:
    for item in ("checkpoint", "prepared.json", "preparation.json", "packed_probes.safetensors"):
        assert (SOURCE_ROOT / arm / item).exists(), f"Missing original evidence: {SOURCE_ROOT / arm / item}"

REPO_DIR = Path("/content/rotquant-native-study/repository")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/CodeHalwell/rotquant.git", str(REPO_DIR)], check=True, timeout=300)
assert not subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip(), "Repository has edits; use a fresh runtime. No automatic reset."
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True, timeout=120)
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "FETCH_HEAD"], text=True).strip()
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", COMMIT], check=True, timeout=30)
for name in ("run_native_gpu_validation.py", "native_gpu_cache.py", "run_rq3_performance_pilot.py", "native_pilot_controls.py", "native_performance_study.py", "native_cuda_diagnostics.py", "run_native_gguf_baseline.py"):
    assert (REPO_DIR / "scripts" / name).exists(), "Pilot code is not present at REPO_REF. Stop before spending on the build."
sys.path.insert(0, str(REPO_DIR))
from scripts.native_pilot_controls import selected_contexts, validate_controls
CONTEXTS = selected_contexts(CONTEXTS)
validate_controls(128, DECODE_STEPS, MEASURED_REPETITIONS,
                  MIN_DECODE_TOKENS_PER_SECOND, MAX_PROCESS_VRAM_MIB)
WORK_DIR = Path("/content/rotquant-native-study/work") / COMMIT[:12]
RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/native_gpu_study") / COMMIT[:12] / RUN_NAME
print({"commit": COMMIT, "results": str(RESULT_ROOT), "cache": str(CACHE_ROOT),
       "arms": ARMS, "active_minutes": ACTIVE_BUDGET_MINUTES,
       "contexts": CONTEXTS, "decode_steps": DECODE_STEPS})

## Steps and checks
1. Verify the original saved model; build/restore the native runtime.
2. Repeat CPU/CUDA operators, tiny W6/W8 models and conversion gates.
3. Pass retained-model parity, then time the original kernel.
4. Check the tiled candidate against canonical arithmetic **and exact
   reference-operator outputs**, including 3/4/5-token tile boundaries.
   Repeat both tiny models and retained-model parity before timing it.
5. Run separate CUDA-event profiles at the smallest selected context:
   rotation, matrix operations, vocabulary head and embedding lookup.
6. Download the pinned BF16 / UD-Q4 controls, verify full file hashes and
   their token-ID maps, then run matched fixed-token timing processes.

The new CUDA code requires **one new build**. The previous runtime cache
cannot validate changed kernel code. Subsequent compatible builds reuse
the private cache. Cold build was about 27 minutes on the prior A100.
Baseline downloads total roughly **11.34 GB**; they stay on local disk.
Drive cache and original checkpoint directories are never overwritten.

Default timing caps are **4 / 6 / 12 minutes** for 128/512/2,048 tokens;
diagnostic caps are doubled, all bounded by the **90 active-minute**
session allowance. Caps stop child processes, **not Colab billing**.

To target just 2,048 tokens, set `CONTEXTS = (2048,)` and a new `RUN_NAME`.
To avoid repeating diagnostics or controls, explicitly set
`RUN_PROFILE = False` or `BASELINES = ()`. Selected timing measurements
and correctness gates remain fresh; no old timing is silently promoted.
After a failure, run the Results cell, download the ZIP and stop the GPU.
Never update the checkout while the driver is running.

In [ ]:
from scripts.colab_runtime import run_live
command = [sys.executable, "-u", str(REPO_DIR / "scripts/run_native_gpu_validation.py"),
           "--output-dir", str(RESULT_ROOT), "--work-dir", str(WORK_DIR),
           "--source-root", str(SOURCE_ROOT), "--persistent-cache-dir", str(CACHE_ROOT),
           "--active-minutes", str(ACTIVE_BUDGET_MINUTES), "--jobs", str(BUILD_JOBS),
           "--performance-study", "--decode-steps", str(DECODE_STEPS),
           "--repetitions", str(MEASURED_REPETITIONS),
           "--min-decode-tps", str(MIN_DECODE_TOKENS_PER_SECOND),
           "--max-vram-mib", str(MAX_PROCESS_VRAM_MIB)]
for arm in ARMS:
    command.extend(["--arm", arm])
for context in CONTEXTS:
    command.extend(["--context", str(context)])
if not RUN_PROFILE:
    command.append("--skip-profile")
if not BASELINES:
    command.append("--no-baselines")
for baseline in BASELINES:
    command.extend(["--baseline", baseline])
run_live(command, "native-performance-study", repo_dir=REPO_DIR,
         log_root=RESULT_ROOT / "launch-logs")

## Results — also run after a stop
Throughput and diagnostic profiles are shown separately. Warmup is
excluded; VRAM is sampled process allocation, not an exact transient peak.
Profiles synchronize CUDA events per custom operation with graphs
disabled. **Do not compare their wall rates to normal throughput.**
Non-custom attention/SSM kernels, host work and copy costs are not
individually attributed by this first profiler; event times do not sum
to complete model wall time.

Reference timing generates greedy decode IDs; candidate and conventional
controls replay those exact IDs, while still performing the same argmax
and model calls. This isolates shape/token-path differences, not output
quality. Token-ID maps are checked without retokenizing multilingual text.
The GGUF controls are **not matched for quality or exact size**. Their
reported size is text-GGUF bytes; the RotQuant non-text sidecar is not in VRAM.

`comparison-*.json` contains ratios only for complete matched,
uninstrumented runs. No automatic kernel or recipe promotion occurs.

In [ ]:
from IPython.display import Markdown, display
from scripts.native_performance_summary import tables
summary_path = RESULT_ROOT / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("Workflow:", summary["status"], "| active minutes:", round(summary["active_minutes"], 1))
    for stage in summary["stages"]:
        print(stage["name"], stage["status"], f"{stage.get('elapsed_seconds', 0):.1f}s")
    if summary.get("error"):
        print("Stopped:", summary["error"])
timing, profiling = tables(RESULT_ROOT)
display(Markdown("### Uninstrumented throughput\n\n" + timing))
display(Markdown("### Diagnostic profiles — not throughput\n\n" + profiling))
archives = sorted(RESULT_ROOT.parent.glob(RUN_NAME + "-reports-*.zip"))
if archives:
    REPORTS_ZIP = archives[-1]
    print("Reports ZIP:", REPORTS_ZIP)
print("DISCONNECT AND DELETE the GPU runtime after downloading your reports.")

## Download and next steps
Download the reports ZIP below, then disconnect/delete the GPU runtime.
Bring back the reports before changing more algorithms or running tasks.
We need measured bottleneck attribution, unchanged parity and a useful
uninstrumented improvement before enabling an optimised kernel by default.

In [ ]:
from google.colab import files
if "REPORTS_ZIP" in globals():
    files.download(str(REPORTS_ZIP))
else:
    print("No report archive found. Run the Results cell first.")